
# 02_decoding_analysis_flexible.ipynb

Flexible MS-AA decoding script.

Supports:
- ANALYSIS_TYPE = "spatial" or "temporal"
- FIT_SCOPE     = "within"  or "across"

Saved factorization conventions:
- spatial AA:
    sXC : T x K
    S   : K x V
    reconstruction: Xhat = sXC @ S

- temporal AA:
    sXC : V x K
    S   : K x T
    reconstruction: Xhat = (sXC @ S).T


In [2]:

ANALYSIS_TYPE = "spatial"   # "spatial" or "temporal"
FIT_SCOPE = "within"         # "within" or "across"

LOAD_DIR = "msaa_flexible_outputs_npz"
OUTPUT_DIR = "msaa_flexible_decoding_outputs"


K_VALUES = [10, 25, 50, 75, 100]
CONDITIONS = ["intact", "word", "rest"]

NFOLDS_DECODE = 2
NREPS_DECODE = 40
TOP_M_LIST = [1, 2, 3, 4, 5]
RNG_SEED = 42
ERRORBAR_TYPE = "std"        # "sem", "std", or "ci95"

SKIP_EXISTING = True

assert ANALYSIS_TYPE in {"spatial", "temporal"}
assert FIT_SCOPE in {"within", "across"}


In [6]:

%matplotlib inline
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

print("Ready.")


Ready.


In [7]:

def to_float_array(x):
    return np.array(x, dtype=float)

def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()

    return {
        "K": int(data["K"]),
        "results_subj": results_subj,
        "condition_labels_str": data["condition_labels_str"].tolist(),
        "condition_codes": data["condition_codes"] if "condition_codes" in data else None,
        "condition_names": data["condition_names"].tolist() if "condition_names" in data else None,
        "analysis_type": data["analysis_type"].tolist() if "analysis_type" in data else ANALYSIS_TYPE,
        "fit_scope": data["fit_scope"].tolist() if "fit_scope" in data else FIT_SCOPE,
    }

def file_for_job(load_dir, analysis_type, fit_scope, K, cond_name=None):
    if fit_scope == "within":
        if cond_name is None:
            raise ValueError("cond_name is required for within-condition loading.")
        fname = f"{analysis_type}AA_within_{cond_name}_K{K}.npz"
    else:
        fname = f"{analysis_type}AA_across_acrossCond_K{K}.npz"
    return os.path.join(load_dir, fname)

def reconstruct_subject(sub, analysis_type, top_indices=None):
    sXC = to_float_array(sub["sXC"])
    S = to_float_array(sub["S"])

    if top_indices is None:
        if analysis_type == "spatial":
            Xhat = sXC @ S
        else:
            Xhat = (sXC @ S).T
    else:
        if analysis_type == "spatial":
            Xhat = sXC[:, top_indices] @ S[top_indices, :]
        else:
            Xhat = (sXC[:, top_indices] @ S[top_indices, :]).T

    Xhat = (Xhat - Xhat.mean(axis=0, keepdims=True)) / (Xhat.std(axis=0, keepdims=True) + 1e-8)
    return Xhat

def build_recon_stack(subjects, analysis_type, top_indices=None):
    return np.stack(
        [reconstruct_subject(sub, analysis_type, top_indices=top_indices) for sub in subjects],
        axis=0
    )

def decoder(corrs):
    out = pd.DataFrame({'rank':[0.0], 'accuracy':[0.0], 'error':[0.0]})
    T = corrs.shape[0]
    for t in range(T):
        decoded_ind = int(np.argmax(corrs[t, :]))
        out.loc[0, 'error'] += np.mean(np.abs(decoded_ind - t)) / T
        out.loc[0, 'accuracy'] += (decoded_ind == t)
        out.loc[0, 'rank'] += np.mean((corrs[t, :] <= corrs[t, t]).astype(int))
    out['error'] /= T
    out['accuracy'] /= T
    out['rank'] /= T
    return out

def get_xval_assignments(ndata, nfolds, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))
    for i in range(1, nfolds):
        inds = np.arange(i * groupsize, min((i + 1) * groupsize, ndata))
        group_assignments[inds] = i
    rng.shuffle(group_assignments)
    return group_assignments

def run_timepoint_decoding(recon_stack, nfolds=2, nreps=20, seed=42):
    N, T, V = recon_stack.shape
    rng = np.random.default_rng(seed)
    all_results = []

    for rep in range(nreps):
        fold_ids = get_xval_assignments(N, nfolds, rng=rng)
        for i in range(nfolds):
            in_mask = (fold_ids == i)
            out_mask = ~in_mask

            in_mean = recon_stack[in_mask].mean(axis=0)
            out_mean = recon_stack[out_mask].mean(axis=0)

            corrs = 1.0 - cdist(in_mean, out_mean, metric='correlation')
            res = decoder(corrs)
            res["rep"] = rep
            res["fold"] = i
            all_results.append(res)

    return pd.concat(all_results, ignore_index=True)

def per_archetype_decoding(subjects, analysis_type, nfolds=2, nreps=20, seed=42):
    K = to_float_array(subjects[0]["sXC"]).shape[1]
    rows = []
    for k in range(K):
        recon_stack = build_recon_stack(subjects, analysis_type, top_indices=[k])
        res = run_timepoint_decoding(recon_stack, nfolds=nfolds, nreps=nreps, seed=seed)
        res["archetype"] = k
        rows.append(res)
    return pd.concat(rows, ignore_index=True)

def run_timepoint_decoding_topm(subjects, analysis_type, ranked_indices, top_ms, nfolds=2, nreps=20, seed=42):
    rows = []
    for m in top_ms:
        top_idx = ranked_indices[:m]
        recon_stack = build_recon_stack(subjects, analysis_type, top_indices=top_idx)
        res = run_timepoint_decoding(recon_stack, nfolds=nfolds, nreps=nreps, seed=seed)
        res["top_m"] = m
        rows.append(res)
    return pd.concat(rows, ignore_index=True)

def summarize_metric_with_error(df, metric_col, errorbar_type="sem", group_cols=None):
    group_cols = [] if group_cols is None else group_cols
    out = df.groupby(group_cols)[metric_col].agg(["mean", "std", "count"]).reset_index()

    if errorbar_type == "sem":
        out["err"] = out["std"] / np.sqrt(out["count"].clip(lower=1))
    elif errorbar_type == "std":
        out["err"] = out["std"]
    elif errorbar_type == "ci95":
        out["err"] = 1.96 * out["std"] / np.sqrt(out["count"].clip(lower=1))
    else:
        raise ValueError("errorbar_type must be 'sem', 'std', or 'ci95'")

    return out

def ensure_output_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def maybe_skip_existing(output_dir):
    required = [
        "full_reconstruction_runs.csv",
        "full_reconstruction_summary.csv",
        "per_archetype_mean_accuracy.csv",
        "topm_runs.csv",
        "topm_summary.csv",
        "best_topm_summary.csv",
        "topm_archetype_identities.csv",
        "rankings.npy",
    ]
    return all((Path(output_dir) / r).exists() for r in required)


In [9]:
print(f"ANALYSIS_TYPE = {ANALYSIS_TYPE}")
print(f"FIT_SCOPE     = {FIT_SCOPE}")
print(f"LOAD_DIR      = {LOAD_DIR}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")

ensure_output_dir(OUTPUT_DIR)

if SKIP_EXISTING and maybe_skip_existing(OUTPUT_DIR):
    print("All expected outputs already exist. Exiting.")
    

full_recon_rows = []
topm_rows = []
per_arch_summary_rows = []
rankings = {}

if FIT_SCOPE == "within":
    for cond_name in CONDITIONS:
        for K in K_VALUES:
            path = file_for_job(LOAD_DIR, ANALYSIS_TYPE, FIT_SCOPE, K, cond_name=cond_name)
            cur = load_msaa_npz(path)
            results_subj = cur["results_subj"]

            print(f"Decoding {ANALYSIS_TYPE}AA | within | {cond_name} | K={K} | n={len(results_subj)}")

            recon_stack = build_recon_stack(results_subj, ANALYSIS_TYPE, top_indices=None)
            res_full = run_timepoint_decoding(recon_stack, NFOLDS_DECODE, NREPS_DECODE, RNG_SEED)
            res_full["condition"] = cond_name
            res_full["K"] = K
            res_full["reconstruction"] = "all_archetypes"
            full_recon_rows.append(res_full)

            per_arch = per_archetype_decoding(results_subj, ANALYSIS_TYPE, NFOLDS_DECODE, NREPS_DECODE, RNG_SEED)
            top_arch = per_arch.groupby("archetype")["accuracy"].mean().sort_values(ascending=False)
            rankings[(cond_name, K)] = top_arch.index.to_numpy()

            pa = top_arch.reset_index()
            pa.columns = ["archetype", "mean_accuracy"]
            pa["condition"] = cond_name
            pa["K"] = K
            per_arch_summary_rows.append(pa)

            res_topm = run_timepoint_decoding_topm(
                results_subj,
                ANALYSIS_TYPE,
                rankings[(cond_name, K)],
                TOP_M_LIST,
                NFOLDS_DECODE,
                NREPS_DECODE,
                RNG_SEED,
            )
            res_topm["condition"] = cond_name
            res_topm["K"] = K
            res_topm["reconstruction"] = "top_m_within_condition"
            topm_rows.append(res_topm)

else:
    for K in K_VALUES:
        path = file_for_job(LOAD_DIR, ANALYSIS_TYPE, FIT_SCOPE, K)
        cur = load_msaa_npz(path)
        results_subj = cur["results_subj"]
        condition_labels_str = np.array(cur["condition_labels_str"])

        print(f"Decoding {ANALYSIS_TYPE}AA | across | K={K} | n={len(results_subj)}")

        for cond_name in sorted(np.unique(condition_labels_str)):
            idx = np.where(condition_labels_str == cond_name)[0]
            sub_cond = [results_subj[i] for i in idx]

            recon_stack = build_recon_stack(sub_cond, ANALYSIS_TYPE, top_indices=None)
            res_full = run_timepoint_decoding(recon_stack, NFOLDS_DECODE, NREPS_DECODE, RNG_SEED)
            res_full["condition"] = cond_name
            res_full["K"] = K
            res_full["reconstruction"] = "all_archetypes"
            full_recon_rows.append(res_full)

        per_arch = per_archetype_decoding(results_subj, ANALYSIS_TYPE, NFOLDS_DECODE, NREPS_DECODE, RNG_SEED)
        top_arch = per_arch.groupby("archetype")["accuracy"].mean().sort_values(ascending=False)
        rankings[K] = top_arch.index.to_numpy()

        pa = top_arch.reset_index()
        pa.columns = ["archetype", "mean_accuracy"]
        pa["condition"] = "across_all_subjects"
        pa["K"] = K
        per_arch_summary_rows.append(pa)

        for cond_name in sorted(np.unique(condition_labels_str)):
            idx = np.where(condition_labels_str == cond_name)[0]
            sub_cond = [results_subj[i] for i in idx]

            res_topm = run_timepoint_decoding_topm(
                sub_cond,
                ANALYSIS_TYPE,
                rankings[K],
                TOP_M_LIST,
                NFOLDS_DECODE,
                NREPS_DECODE,
                RNG_SEED,
            )
            res_topm["condition"] = cond_name
            res_topm["K"] = K
            res_topm["reconstruction"] = "top_m_across_condition_ranked"
            topm_rows.append(res_topm)

full_recon_df = pd.concat(full_recon_rows, ignore_index=True)
topm_df = pd.concat(topm_rows, ignore_index=True)
per_arch_summary_df = pd.concat(per_arch_summary_rows, ignore_index=True)

full_summary = summarize_metric_with_error(
    full_recon_df, "accuracy", ERRORBAR_TYPE, ["condition", "K", "reconstruction"]
)
topm_summary = summarize_metric_with_error(
    topm_df, "accuracy", ERRORBAR_TYPE, ["condition", "K", "reconstruction", "top_m"]
)

best_topm_rows = []
for cond_name in CONDITIONS:
    for K in K_VALUES:
        sub = topm_summary[(topm_summary["condition"] == cond_name) & (topm_summary["K"] == K)]
        if len(sub):
            best_topm_rows.append(sub.loc[sub["mean"].idxmax()].copy())
best_topm_df = pd.DataFrame(best_topm_rows)

topm_archetypes_rows = []
if FIT_SCOPE == "within":
    for cond_name in CONDITIONS:
        for K in K_VALUES:
            ranked = rankings[(cond_name, K)]
            for m in TOP_M_LIST:
                topm_archetypes_rows.append({
                    "condition": cond_name,
                    "K": K,
                    "top_m": m,
                    "selected_archetypes": list(ranked[:m]),
                    "selected_archetypes_str": str(list(ranked[:m])),
                })
else:
    for K in K_VALUES:
        ranked = rankings[K]
        for cond_name in CONDITIONS:
            for m in TOP_M_LIST:
                topm_archetypes_rows.append({
                    "condition": cond_name,
                    "K": K,
                    "top_m": m,
                    "selected_archetypes": list(ranked[:m]),
                    "selected_archetypes_str": str(list(ranked[:m])),
                })

topm_archetypes_df = pd.DataFrame(topm_archetypes_rows)

full_recon_df.to_csv(os.path.join(OUTPUT_DIR, "full_reconstruction_runs.csv"), index=False)
topm_df.to_csv(os.path.join(OUTPUT_DIR, "topm_runs.csv"), index=False)
per_arch_summary_df.to_csv(os.path.join(OUTPUT_DIR, "per_archetype_mean_accuracy.csv"), index=False)
full_summary.to_csv(os.path.join(OUTPUT_DIR, "full_reconstruction_summary.csv"), index=False)
topm_summary.to_csv(os.path.join(OUTPUT_DIR, "topm_summary.csv"), index=False)
best_topm_df.to_csv(os.path.join(OUTPUT_DIR, "best_topm_summary.csv"), index=False)
topm_archetypes_df.to_csv(os.path.join(OUTPUT_DIR, "topm_archetype_identities.csv"), index=False)
np.save(os.path.join(OUTPUT_DIR, "rankings.npy"), rankings, allow_pickle=True)

print("\nSaved outputs to:", OUTPUT_DIR)
print("Done.")

ANALYSIS_TYPE = spatial
FIT_SCOPE     = within
LOAD_DIR      = msaa_flexible_outputs_npz
OUTPUT_DIR    = msaa_flexible_decoding_outputs
Decoding spatialAA | within | intact | K=10 | n=36
Decoding spatialAA | within | intact | K=25 | n=36
Decoding spatialAA | within | intact | K=50 | n=36
Decoding spatialAA | within | intact | K=75 | n=36
Decoding spatialAA | within | intact | K=100 | n=36
Decoding spatialAA | within | word | K=10 | n=36
Decoding spatialAA | within | word | K=25 | n=36
Decoding spatialAA | within | word | K=50 | n=36
Decoding spatialAA | within | word | K=75 | n=36
Decoding spatialAA | within | word | K=100 | n=36
Decoding spatialAA | within | rest | K=10 | n=36
Decoding spatialAA | within | rest | K=25 | n=36
Decoding spatialAA | within | rest | K=50 | n=36
Decoding spatialAA | within | rest | K=75 | n=36
Decoding spatialAA | within | rest | K=100 | n=36

Saved outputs to: msaa_flexible_decoding_outputs
Done.
